# Lección 5.1 — Regresión lineal: la recta que mejor se ajusta

> **Módulo 5 · Modelos clásicos supervisados** · Nivel 2 — Aprendiz
> ⏱️ Tiempo estimado: 100–120 min (lección con ejercicios intercalados ≈ 75 min · ejercicios finales ≈ 30 min)
> 📋 Requisitos previos: Lección 3.2 (función de pérdida), Lección 3.3 (descenso de gradiente), Lección 4.1 (MAE, RMSE, R²)

### 🎯 Objetivos
Al terminar esta lección podrás:
- Explicar con una analogía qué hace una regresión lineal y qué significan su **intercepto** y sus **coeficientes**.
- Describir **cómo aprende**: buscando la recta con el menor error cuadrático.
- Predecir qué le pasa al modelo con relaciones curvas, valores atípicos y datos fuera de rango.
- Usar sus "perillas" indirectas: crear características (grado del polinomio) y regularizar (`alpha`).
- Reconocer cuándo una regresión lineal es una buena elección y cuándo no.

### 🧭 Cómo usar este notebook
| Icono | Significa |
|---|---|
| 📖 | Lee: explicación del concepto |
| ▶️ | Ejecuta la celda y observa el resultado |
| 🔮 | **Predice** antes de ejecutar: escribe tu apuesta y luego compruébala |
| ✋ | **Tu turno**: ejercicio. Las respuestas están ocultas en "👀 Ver respuesta" |
| ✏️ | Modifica un valor en el código y vuelve a ejecutar |

> 💡 No necesitas saber programar. El código está para que **veas** los conceptos en acción; solo tendrás que cambiar números.

## ▶️ Preparación
Ejecuta esta celda una vez (`Shift + Enter`). Carga las herramientas que usaremos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

plt.rcParams["figure.figsize"] = (8, 4.5)
print("✅ Todo listo")

---
## 1. 📖 La intuición: la tarifa del taxi

Un taxi te cobra así:

```
  Precio = banderazo  +  tarifa por km × kilómetros
         =    $40     +      $10       ×    km
```

- El **banderazo** ($40) lo pagas aunque no avances: es el **punto de partida**.
- La **tarifa por km** ($10) dice **cuánto sube** el precio por cada kilómetro extra.

Si dibujas precio contra kilómetros, obtienes una **recta**. Una regresión lineal hace el viaje al revés: **mira muchos viajes ya cobrados y descubre el banderazo y la tarifa** que mejor los explican.

### ✋ Ejercicio 1 — Calentamiento
1. Con esa tarifa, ¿cuánto cuesta un viaje de **7 km**?
2. Otro taxi cobra $70 por 3 km y $110 por 7 km. ¿Cuánto cobra por km? ¿Y de banderazo?

✍️ *Tu respuesta:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. $40 + $10 × 7 = **$110**.
2. 4 km más cuestan $40 más → **$10 por km**. Si 3 km cuestan $70 y $30 son por los km, el banderazo es **$40**. (¡Es el mismo taxi!)

Acabas de hacer, a mano, lo que hace una regresión lineal: deducir el **punto de partida** y la **pendiente** a partir de ejemplos.
</details>

---
## 2. 📖 El concepto: una recta con dos perillas

Con una sola característica, una regresión lineal es:

$$\text{predicción} = b + w \times x$$

| Símbolo | Nombre | En el taxi |
|---|---|---|
| $x$ | Característica (*feature*) | Kilómetros |
| $b$ | **Intercepto** (*intercept*): la predicción cuando $x = 0$ | Banderazo |
| $w$ | **Coeficiente** o pendiente (*coefficient*): cuánto cambia la predicción por cada unidad de $x$ | Tarifa por km |

$b$ y $w$ son los **parámetros** del modelo (Lección 1.2): las perillas que el modelo ajusta solo.

Vamos con datos: una inmobiliaria tiene 120 departamentos vendidos con sus m² y su precio (en miles de $).

In [ ]:
# ▶️ Generamos ventas ficticias de departamentos
rng = np.random.default_rng(0)
m2 = rng.uniform(30, 150, 120)
precio = 500 + 20 * m2 + rng.normal(0, 150, 120)     # regla "real" + ruido (el modelo NO la conoce)

deptos = pd.DataFrame({"m2": m2.round(), "precio_miles": precio.round()})
plt.scatter(deptos.m2, deptos.precio_miles, s=15)
plt.xlabel("m²"); plt.ylabel("Precio (miles de $)"); plt.title("Departamentos vendidos"); plt.show()

Le pedimos a la regresión lineal que encuentre la recta. En scikit-learn, el intercepto se guarda en `intercept_` y los coeficientes en `coef_`.

In [ ]:
modelo = LinearRegression().fit(deptos[["m2"]], deptos.precio_miles)
print(f"Intercepto (b): {modelo.intercept_:.0f}")
print(f"Coeficiente (w): {modelo.coef_[0]:.1f}  por cada m²")

plt.scatter(deptos.m2, deptos.precio_miles, s=15)
xs = np.linspace(30, 150, 2)
plt.plot(xs, modelo.intercept_ + modelo.coef_[0] * xs, "r-", lw=2, label="Recta aprendida")
plt.xlabel("m²"); plt.ylabel("Precio (miles de $)"); plt.legend(); plt.show()

### ✋ Ejercicio 2 — Sé el modelo
Usando el intercepto y el coeficiente que se imprimieron arriba:
1. ¿Qué significa el coeficiente, **en palabras de un vendedor**?
2. Calcula **a mano** el precio que predice para un departamento de **80 m²**.

✍️ *Tus respuestas:*

Luego ejecuta la celda de abajo para comprobar el cálculo.

In [ ]:
nuevo = pd.DataFrame({"m2": [80]})
print(f"Predicción para 80 m²: {modelo.predict(nuevo)[0]:,.0f} miles de $")

<details><summary><b>👀 Ver respuesta</b></summary>

1. "Cada m² extra suma unos **20 mil pesos** al precio" (el modelo encontró ≈19.7, muy cerca de la regla real de 20).
2. ≈ 516 + 19.7 × 80 ≈ **2,090** miles de $ (la celda imprime 2,092).

Fíjate: el modelo **no conocía** la regla real (500 + 20 × m²) y la recuperó casi exacta solo mirando ejemplos.
</details>

### 📖 Con varias características: una suma ponderada
Casi nunca hay una sola característica. Con varias, la idea es la misma: **cada característica tiene su propio coeficiente** y se suman.

$$\text{precio} = b + w_1 \cdot \text{m}^2 + w_2 \cdot \text{recámaras} + w_3 \cdot \text{antigüedad}$$

Ya no es una recta sino un plano (o algo con más dimensiones), pero se interpreta igual: **cada coeficiente dice cuánto cambia la predicción si esa característica sube 1 unidad y todas las demás se quedan igual.**

In [ ]:
n = 300
casas = pd.DataFrame({
    "m2":         rng.uniform(40, 200, n).round(),
    "recamaras":  rng.integers(1, 5, n),
    "antiguedad": rng.integers(0, 40, n),           # años
})
casas["precio_miles"] = (400 + 18 * casas.m2 + 60 * casas.recamaras
                         - 8 * casas.antiguedad + rng.normal(0, 150, n)).round()

X_casas = casas[["m2", "recamaras", "antiguedad"]]
modelo_casas = LinearRegression().fit(X_casas, casas.precio_miles)
print(f"Intercepto: {modelo_casas.intercept_:.0f}")
print(pd.Series(modelo_casas.coef_, index=X_casas.columns).round(1))

### ✋ Ejercicio 3 — Lee los coeficientes
1. ¿Qué característica **baja** el precio? ¿Cuánto por año?
2. Dos casas idénticas, pero una tiene una recámara más. ¿Cuánto más cuesta según el modelo?
3. ¿Puedes concluir que "los m² son la característica **menos** importante" porque su coeficiente es menor que el de las recámaras?

✍️ *Tus respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. La **antigüedad**: coeficiente negativo (≈ −8). Cada año resta unos 8 mil pesos.
2. Unos **67 mil pesos** más (el coeficiente de `recamaras`; la regla real usaba 60, el ruido lo movió un poco).
3. **No.** Los coeficientes dependen de las **unidades**. Un m² es poquito; una recámara es mucho. Pasar de 60 a 160 m² suma ≈ 1,800 mil, ¡mucho más que cualquier cambio en recámaras! Para comparar la importancia hay que poner las características en la misma escala (lo verás en la Sección 6).
</details>

---
## 3. 📖 Cómo aprende: la recta con el menor error cuadrático

Hay infinitas rectas posibles. ¿Cuál es "la mejor"? Para cada departamento medimos el **error** (o **residuo**): la distancia vertical entre el precio real y la recta. Luego:

1. Elevamos cada error **al cuadrado** (así los negativos no cancelan a los positivos y los errores grandes pesan mucho más).
2. Los promediamos: es el **error cuadrático medio** (MSE, Lección 3.2).
3. La mejor recta es la que tiene el **MSE más pequeño**. Por eso al método se le llama **mínimos cuadrados**.

> 💡 **¿Y el descenso de gradiente (Lección 3.3)?** Funcionaría: el excursionista bajaría la montaña del MSE hasta el fondo. Pero para la regresión lineal existe un atajo: una fórmula que da el fondo **directamente**, sin pasos. Eso es lo que usa `LinearRegression`. Y como la "montaña" del MSE tiene forma de tazón, **solo hay un fondo**: no hay mínimos locales (Lección 3.6).

### 🔮 Ejercicio 4 — ¿Cuál recta gana?
Tres vendedores proponen su propia regla para los departamentos:

- **A:** precio = 0 + 27 × m²
- **B:** precio = 1,000 + 15 × m²
- **C:** precio = 500 + 20 × m²

Antes de ejecutar: ¿cuál tendrá el menor error cuadrático medio? ¿Le ganará alguna a la recta de `LinearRegression`?

✍️ *Mi apuesta:*

In [ ]:
def mse(b, w):
    error = deptos.precio_miles - (b + w * deptos.m2)
    return (error ** 2).mean()

candidatas = {"A": (0, 27), "B": (1_000, 15), "C": (500, 20),
              "LinearRegression": (modelo.intercept_, modelo.coef_[0])}
# ✏️ añade tu propia recta, por ejemplo:  "Yo": (300, 22),
for nombre, (b, w) in candidatas.items():
    print(f"{nombre:>16}: b={b:>6.0f}  w={w:>5.1f}  →  MSE = {mse(b, w):>9,.0f}")

<details><summary><b>👀 Ver respuesta</b></summary>

**C** es la mejor de las tres propuestas (es justo la regla con la que generamos los datos), pero **`LinearRegression` le gana por poquito**. ¿Cómo puede ganarle a la regla "real"? Porque busca la recta que mejor se ajusta a **estos** 120 ejemplos, ruido incluido. Esa pequeña diferencia es el modelo acomodándose al ruido: con pocos datos es un primer aviso de sobreajuste (Lección 4.4).

Ninguna recta, por definición, puede tener menor MSE que la de `LinearRegression` en estos datos: esa es su definición de "la mejor".
</details>

---
## 4. 📖 ¿Qué pasa si…?

La regresión lineal tiene **supuestos**: cree que la relación es una recta y que los errores son "ruido razonable". Veamos qué ocurre cuando los datos no cooperan.

### 4.1 …la relación es curva
El consumo de gasolina de un auto es alto a muy baja velocidad, mínimo a velocidad media y vuelve a subir a alta velocidad: tiene forma de **U**.

### 🔮 Ejercicio 5 — Una recta para una U
Vamos a ajustar una recta a esos datos. ¿Qué $R^2$ esperas (Lección 4.1: 1 = perfecto, 0 = igual que predecir el promedio)?

✍️ *Mi apuesta:*

In [ ]:
velocidad = rng.uniform(20, 140, 150)
consumo = 4 + 0.0012 * (velocidad - 80) ** 2 + rng.normal(0, 0.4, 150)    # litros / 100 km
V = velocidad.reshape(-1, 1)

recta = LinearRegression().fit(V, consumo)
print(f"R² de la recta: {r2_score(consumo, recta.predict(V)):.2f}")

xs = np.linspace(20, 140, 200).reshape(-1, 1)
plt.scatter(velocidad, consumo, s=12)
plt.plot(xs, recta.predict(xs), "r-", lw=2, label="Recta")
plt.xlabel("Velocidad (km/h)"); plt.ylabel("Consumo (L/100 km)"); plt.legend(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

$R^2$ ≈ **0.02**, prácticamente 0: la recta es casi plana y no explica nada. Es **subajuste** (Lección 4.4): el modelo es demasiado simple para la forma de los datos. La recta no "ve" la U porque solo sabe dibujar rectas.
</details>

### 📖 El truco: inventar características
La regresión lineal es lineal **en sus coeficientes**, no necesariamente en los datos. Si le damos una nueva característica, **velocidad²**, la suma ponderada $b + w_1 \cdot v + w_2 \cdot v^2$ ya puede dibujar una curva. Esto es **ingeniería de características** (Lección 2.4), y `PolynomialFeatures` lo hace por nosotros.

In [ ]:
curva = make_pipeline(PolynomialFeatures(degree=2), LinearRegression()).fit(V, consumo)
print(f"R² con velocidad²: {r2_score(consumo, curva.predict(V)):.2f}")

plt.scatter(velocidad, consumo, s=12)
plt.plot(xs, curva.predict(xs), "g-", lw=2, label="Recta + velocidad²")
plt.xlabel("Velocidad (km/h)"); plt.ylabel("Consumo (L/100 km)"); plt.legend(); plt.show()

### 4.2 …hay un valor atípico
Alguien capturó mal un departamento: 60 m² con un precio de **9,000** (en lugar de ~1,700).

### 🔮 Ejercicio 6 — Un solo dato malo
Hay 120 datos buenos y **1** malo. ¿Crees que la recta cambiará mucho, poco o nada? ¿Hacia dónde se moverá?

✍️ *Mi apuesta:*

In [ ]:
con_error = pd.concat([deptos, pd.DataFrame({"m2": [60], "precio_miles": [9_000]})])
modelo_error = LinearRegression().fit(con_error[["m2"]], con_error.precio_miles)

print(f"Sin el error: b = {modelo.intercept_:>5.0f}   w = {modelo.coef_[0]:.1f}")
print(f"Con el error: b = {modelo_error.intercept_:>5.0f}   w = {modelo_error.coef_[0]:.1f}")

plt.scatter(con_error.m2, con_error.precio_miles, s=15)
plt.plot(xs, modelo.intercept_ + modelo.coef_[0] * xs, "g-", lw=2, label="Sin el error")
plt.plot(xs, modelo_error.intercept_ + modelo_error.coef_[0] * xs, "r--", lw=2, label="Con el error")
plt.xlabel("m²"); plt.ylabel("Precio (miles de $)"); plt.legend(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

**Un solo dato** sube el intercepto unos 200 mil pesos (516 → 721) y baja la pendiente (de ≈19.7 a ≈18.2). La recta entera se desplaza hacia el punto raro.

¿Por qué tanto? Porque los errores se elevan **al cuadrado**: un error de 7,000 pesa como 49 millones, más que todos los demás juntos. La regresión lineal es **sensible a los atípicos**. Por eso conviene revisar los datos antes (Lección 2.2) o usar una pérdida menos sensible, como la MAE (Lección 3.2).
</details>

### 4.3 …le preguntas fuera del rango que vio
En la Lección 5.4 viste que un árbol **no extrapola**: a una mansión de 1,000 m² le daba el precio de una casa de 300 m². La regresión lineal sí extrapola: **sigue la recta**. ¿Siempre es buena idea?

### 🔮 Ejercicio 7 — El niño gigante
Un pediatra registra la estatura de niños de **2 a 12 años**. Entrenamos una regresión lineal y le preguntamos por una persona de **40 años**. ¿Qué estatura crees que predice?

✍️ *Mi apuesta:*

In [ ]:
edad = rng.uniform(2, 12, 100)
estatura = 80 + 6 * edad + rng.normal(0, 4, 100)       # cm: los niños crecen ~6 cm por año
modelo_ninos = LinearRegression().fit(edad.reshape(-1, 1), estatura)

for e in [5, 10, 40]:
    print(f"Edad {e:>2} años → estatura predicha: {modelo_ninos.predict([[e]])[0]:.0f} cm")

<details><summary><b>👀 Ver respuesta</b></summary>

¡Unos **3.3 metros** (327 cm)! Dentro del rango (5 y 10 años) las predicciones son razonables, pero a los 40 años el modelo **sigue la recta** como si siguiéramos creciendo 6 cm por año toda la vida.

Moraleja: extrapolar es **una virtud y un peligro**. Funciona si la tendencia realmente continúa (el precio por m² de la Lección 5.4), y falla de forma absurda si no (la estatura). **El modelo no lo sabe; tú sí.**
</details>

---
## 5. 📖 Hiperparámetros: las perillas indirectas

La `LinearRegression` básica casi no tiene hiperparámetros: encuentra la mejor recta y ya. Pero en la práctica hay **dos perillas** que tú controlas:

| Perilla | Qué controla | Si la **subes** ⬆️ | Si la **bajas** ⬇️ | Punto de partida |
|---|---|---|---|---|
| `degree` (de `PolynomialFeatures`) | Qué tan curva puede ser la predicción | Más flexible → riesgo de **sobreajuste** | Más rígida → riesgo de **subajuste** | 1 (recta); 2–3 si ves curvatura |
| `alpha` (en `Ridge` / `Lasso`) | **Regularización**: castigo a los coeficientes grandes | Coeficientes más pequeños, modelo más simple y estable | Se parece a la regresión lineal normal | 1.0 (ajustar con validación cruzada) |
| `fit_intercept` | Si se aprende el intercepto $b$ | — | — | `True` (casi siempre) |

> 📖 **Regularización en una frase:** `Ridge` es una regresión lineal que, además de reducir el error, **castiga los coeficientes grandes**. Así evita curvas exageradas. La verás a fondo en la Lección 7.1 (L1/Lasso, L2/Ridge, Elastic Net).

### ✏️ Ejercicio 8 — El grado del polinomio
Tenemos solo **20 puntos** con ruido. La celda ajusta polinomios de grado 1, 3 y 15. Antes de ejecutar: ¿cuál sobreajusta? ¿Cuál subajusta?

✍️ *Mis predicciones:*

In [ ]:
rng_p = np.random.default_rng(3)
x_poco = np.sort(rng_p.uniform(0, 1, 20)).reshape(-1, 1)
y_poco = np.sin(2 * np.pi * x_poco).ravel() + rng_p.normal(0, 0.25, 20)
x_nuevo = rng_p.uniform(0, 1, 200).reshape(-1, 1)                  # datos nuevos para "prueba"
y_nuevo = np.sin(2 * np.pi * x_nuevo).ravel() + rng_p.normal(0, 0.25, 200)

GRADOS = [1, 3, 15]          # ✏️ prueba otros grados
xs01 = np.linspace(0, 1, 300).reshape(-1, 1)
fig, axes = plt.subplots(1, len(GRADOS), figsize=(4 * len(GRADOS), 3.5))
for ax, g in zip(axes, GRADOS):
    m = make_pipeline(PolynomialFeatures(g), StandardScaler(), LinearRegression()).fit(x_poco, y_poco)
    ax.scatter(x_poco, y_poco, s=15); ax.plot(xs01, m.predict(xs01), "r-")
    ax.set_ylim(-2, 2)
    ax.set_title(f"grado={g}\nR² train {r2_score(y_poco, m.predict(x_poco)):.2f} · "
                 f"test {r2_score(y_nuevo, m.predict(x_nuevo)):.2f}", fontsize=10)
plt.tight_layout(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- **Grado 1** → una recta inclinada para una ola: solo capta la tendencia general (R² de prueba ≈0.45). **Subajuste.**
- **Grado 3** → captura la ola: buen R² en entrenamiento (0.92) **y** en prueba (≈0.78).
- **Grado 15** → el mejor R² de entrenamiento (0.94), pero la curva se retuerce para pasar por cada punto y en prueba se desploma (R² ≈ −57: ¡muchísimo peor que predecir el promedio!). **Sobreajuste.**

Es la misma "U" de la Lección 4.4, ahora con la perilla `degree`.
</details>

### 🔮 Ejercicio 9 — La regularización al rescate
Mantenemos el polinomio de **grado 15** (el que sobreajustaba), pero cambiamos `LinearRegression` por `Ridge` con distintos `alpha`. ¿Qué esperas que pase con la curva al subir `alpha`? ¿Habrá un `alpha` demasiado grande?

✍️ *Mi predicción:*

In [ ]:
ALPHAS = [1e-08, 0.01, 100]     # ✏️ prueba otros valores
fig, axes = plt.subplots(1, len(ALPHAS), figsize=(4 * len(ALPHAS), 3.5))
for ax, a in zip(axes, ALPHAS):
    m = make_pipeline(PolynomialFeatures(15), StandardScaler(), Ridge(alpha=a)).fit(x_poco, y_poco)
    ax.scatter(x_poco, y_poco, s=15); ax.plot(xs01, m.predict(xs01), "r-")
    ax.set_ylim(-2, 2)
    ax.set_title(f"grado 15, alpha={a}\nR² train {r2_score(y_poco, m.predict(x_poco)):.2f} · "
                 f"test {r2_score(y_nuevo, m.predict(x_nuevo)):.2f}", fontsize=10)
plt.tight_layout(); plt.show()

<details><summary><b>👀 Ver respuesta</b></summary>

- `alpha=1e-08` (casi cero) → casi no castiga: la curva se sigue retorciendo y el R² de prueba es negativo (≈ −4.8). Sigue el **sobreajuste**.
- `alpha=0.01` → la curva se suaviza y captura la ola: el mejor R² de prueba de los tres (≈0.79), ¡con el mismo polinomio de grado 15!
- `alpha=100` → castiga tanto que la curva se aplana (R² ≈0.47 en entrenamiento y ≈0.40 en prueba): **subajuste**.

Mismo patrón que siempre: **demasiado poco castigo sobreajusta, demasiado castigo subajusta**. El `alpha` justo se elige con validación cruzada (Lección 4.6).
</details>

---
## 6. 📖 Errores comunes y señales de alerta

| 🚨 Síntoma | Causa probable | Qué probar |
|---|---|---|
| $R^2$ bajo en entrenamiento **y** en prueba | La relación no es una recta → subajuste | Graficar los residuos; crear características (², logaritmos, interacciones) |
| Los residuos forman una curva o un embudo | El modelo no captura la forma de los datos | Transformar características o el objetivo; probar un modelo no lineal |
| Un coeficiente cambia muchísimo al quitar unas pocas filas | Valores atípicos | Revisar y limpiar los datos (Lección 2.2) |
| Coeficientes enormes, con signos raros, que se contradicen | Características muy correlacionadas entre sí (*multicolinealidad*) | Quitar una de las dos o usar `Ridge` |
| Predicciones absurdas fuera del rango de entrenamiento | La recta se extrapola sin límite | No predecir fuera del rango, o advertirlo |
| "Esta variable es la más importante porque su coeficiente es el más grande" | Comparar coeficientes con **unidades distintas** | Estandarizar las características antes de comparar |

El último error es el más frecuente al **explicar** un modelo. Comprobémoslo.

### 🔮 Ejercicio 10 — Mismo modelo, otras unidades
Entrenamos el modelo de casas dos veces: una con el tamaño en **m²** y otra en **cm²** (1 m² = 10,000 cm²).
1. ¿Cambiarán las **predicciones**?
2. ¿Cambiará el **coeficiente** del tamaño?

✍️ *Mis predicciones:*

In [ ]:
X_cm = X_casas.assign(m2=X_casas.m2 * 10_000).rename(columns={"m2": "cm2"})
modelo_cm = LinearRegression().fit(X_cm, casas.precio_miles)

print("Coeficientes con m² :", modelo_casas.coef_.round(4))
print("Coeficientes con cm²:", modelo_cm.coef_.round(4))
print("¿Mismas predicciones?", np.allclose(modelo_casas.predict(X_casas), modelo_cm.predict(X_cm)))

# Para comparar importancias, estandarizamos: todas las características en "desviaciones estándar"
estandar = make_pipeline(StandardScaler(), LinearRegression()).fit(X_casas, casas.precio_miles)
print("\nCoeficientes estandarizados (cuánto cambia el precio si la característica sube 1 desviación estándar):")
print(pd.Series(estandar[-1].coef_, index=X_casas.columns).round(0))

<details><summary><b>👀 Ver respuesta</b></summary>

1. **No.** Las predicciones son idénticas (`True`).
2. **Sí:** el coeficiente del tamaño se vuelve **10,000 veces más pequeño** (de 18 a 0.0018), porque ahora cada unidad es diminuta.

Con los coeficientes **estandarizados** vemos la importancia real: el tamaño (≈ 805) pesa muchísimo más que las recámaras (≈ 73) o la antigüedad (≈ −91). Justo lo contrario de lo que sugerían los coeficientes "crudos" del Ejercicio 3.

> 💡 Estandarizar **no cambia** las predicciones de una `LinearRegression`, pero es **imprescindible** con `Ridge`/`Lasso` (el castigo trataría distinto a cada variable) y, como verás, con KNN y SVM.
</details>

---
## 7. 📋 Ficha del modelo

| | Regresión lineal |
|---|---|
| **Tipo** | Supervisado: regresión |
| **Intuición** | La recta (o plano) que mejor pasa entre los puntos: un punto de partida más una suma ponderada de las características |
| **Cómo aprende** | Busca los coeficientes que minimizan el error cuadrático medio (mínimos cuadrados), con una fórmula directa o con descenso de gradiente |
| **Hiperparámetros clave** | Casi ninguno; las perillas reales son las características que creas (`degree`) y la regularización (`alpha` en `Ridge`/`Lasso`) |
| **✅ Fortalezas** | Muy interpretable (cada coeficiente se lee) · Rapidísima · Pocos datos bastan · Extrapola la tendencia · Excelente línea base |
| **❌ Debilidades** | Solo capta relaciones lineales (salvo que crees características) · Sensible a atípicos · Coeficientes engañosos si hay variables correlacionadas · Extrapola aunque no deba |
| **Úsalo cuando…** | Quieres una primera línea base, necesitas explicar el efecto de cada variable o la relación es aproximadamente lineal |
| **Evítalo cuando…** | La relación es muy no lineal o con interacciones complejas que no sabes construir a mano (usa árboles o ensambles, M6) |

## 8. 🧠 Resumen en 5 puntos
1. Una regresión lineal predice con un **intercepto** más una **suma ponderada** de las características.
2. Aprende buscando los coeficientes con el **menor error cuadrático** (mínimos cuadrados): hay un único mejor resultado.
3. Cada coeficiente se lee como "cuánto cambia la predicción si esta variable sube 1 unidad y las demás no cambian", pero su **tamaño depende de las unidades**.
4. Sus puntos débiles son las **curvas** (se arreglan creando características), los **atípicos** y la **extrapolación** sin control.
5. Sus perillas son el **grado** de las características y la **regularización** (`alpha`): demasiado de lo primero sobreajusta, demasiado de lo segundo subajusta.

---
## 9. 🏋️ Ejercicios finales

Ya practicaste cada idea por separado. Ahora toca **combinarlas**. Estos ejercicios son más largos (≈30 min).

### 9.1 Repaso rápido (opción múltiple)

**1.** Un modelo dice: `salario = 9,000 + 1,500 × años_experiencia`. ¿Qué significa el 1,500?
- a) El salario mínimo
- b) Cuánto sube el salario predicho por cada año extra de experiencia
- c) El error promedio del modelo
- d) El salario de alguien con 1 año de experiencia

**2.** ¿Qué busca minimizar `LinearRegression`?
- a) El número de puntos mal clasificados
- b) La suma de los errores (sin elevar)
- c) El promedio de los errores al cuadrado
- d) La pendiente de la recta

**3.** Tu regresión lineal tiene $R^2$ = 0.15 en entrenamiento y 0.13 en prueba. Los residuos forman una U. ¿Qué tiene **más** sentido?
- a) Subir `alpha` de `Ridge`
- b) Agregar una característica al cuadrado
- c) Quitar datos de entrenamiento
- d) Nada: el modelo está bien

**4.** Un polinomio de grado 12 tiene $R^2$ = 0.99 en entrenamiento y 0.40 en prueba. ¿Qué perilla moverías primero?
- a) Bajar `alpha` · b) Subir `degree` · c) Subir `alpha` (o bajar `degree`) · d) Quitar el intercepto

**5.** Cambias una característica de kilómetros a metros. ¿Qué ocurre en una `LinearRegression`?
- a) Las predicciones cambian · b) El coeficiente de esa característica se divide entre 1,000 · c) El $R^2$ empeora · d) Nada cambia

✍️ *Mis respuestas:* 1 → _ · 2 → _ · 3 → _ · 4 → _ · 5 → _

<details><summary><b>👀 Ver respuestas</b></summary>

1. **b)** Es la pendiente: el cambio en la predicción por cada unidad de la característica. El 9,000 es el intercepto (la predicción con 0 años).
2. **c)** Es el error cuadrático medio (mínimos cuadrados). a) sería clasificación.
3. **b)** $R^2$ bajo en ambos + residuos en U = subajuste por una relación curva. a) simplificaría aún más.
4. **c)** Brecha enorme = sobreajuste. Más regularización o un polinomio más simple.
5. **b)** Cada unidad es ahora 1,000 veces más pequeña, así que el coeficiente se ajusta. Las predicciones y el $R^2$ no cambian.
</details>

### 9.2 Preguntas para pensar (sin ejecutar)

**A.** Un analista dice: *"En mi modelo de ventas, el coeficiente de 'número de vendedores' es negativo, así que contratar vendedores **reduce** las ventas"*. Las tiendas grandes, que venden más, también tienen más vendedores. Da al menos **dos razones** por las que su conclusión podría estar equivocada.

**B. Caso de análisis 💼** Una aseguradora quiere predecir el costo anual de un siniestro. El equipo duda entre:
- **Opción A:** una regresión lineal con $R^2$ = 0.71 en validación cruzada.
- **Opción B:** un árbol de regresión con $R^2$ = 0.74.

El 5 % de los clientes nuevos tendrá valores **fuera del rango** visto en entrenamiento (vehículos más caros que cualquiera del histórico). ¿Qué riesgos tiene cada opción con esos clientes? ¿Cuál elegirías?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuestas</b></summary>

**A.** Algunas razones:
- **Multicolinealidad:** si "vendedores" está muy correlacionado con otra variable del modelo (p. ej. "m² de la tienda"), el modelo puede repartir el efecto de forma rara y dar signos contraintuitivos.
- **Correlación no es causalidad:** un coeficiente describe una asociación en los datos, "manteniendo lo demás igual", no el efecto de una **intervención**. Quizá se contrata más gente en tiendas que venden poco para intentar rescatarlas.
- **Atípicos o pocas tiendas** pueden mover un coeficiente.

**B.** No hay una única respuesta correcta; importa el razonamiento.
- El **árbol** no extrapola: a un vehículo más caro que cualquiera del histórico le dará el costo de los más caros que vio → **subestimará** sistemáticamente esos siniestros.
- La **regresión lineal** extrapola la tendencia: puede acertar si la relación sigue siendo lineal, o exagerar si no.
- Un buen argumento: con un 5 % de casos fuera de rango y una diferencia de $R^2$ pequeña, la lineal es más segura y explicable, **siempre que** se revise cómo se comporta en los casos extremos (o se marquen para revisión humana).
</details>

### 9.3 🧩 Práctica integradora: "¿Dónde invierto en publicidad?"

Una marca de bebidas te entrega 200 semanas de datos: cuánto invirtió en **TV**, **radio** y **periódico** (miles de $) y cuánto **vendió** (miles de unidades). Quieren saber **qué medio funciona** y **cuánto venderán** el próximo mes.

Lo harás en 6 pasos cortos. Cada uno tiene su respuesta: **no avances sin revisarla**.

**Paso 1 ▶️ — Mira los datos**

In [ ]:
rng_ad = np.random.default_rng(11)
n_ad = 200
ads = pd.DataFrame({
    "tv":        rng_ad.uniform(0, 300, n_ad).round(1),
    "radio":     rng_ad.uniform(0, 50, n_ad).round(1),
    "periodico": rng_ad.uniform(0, 100, n_ad).round(1),
})
ads["ventas"] = (2 + 0.9 * np.sqrt(ads.tv) + 0.12 * ads.radio + rng_ad.normal(0, 1, n_ad)).round(1)

X_ad = ads[["tv", "radio", "periodico"]]
y_ad = ads["ventas"]
X_ad_tr, X_ad_te, y_ad_tr, y_ad_te = train_test_split(X_ad, y_ad, test_size=0.25, random_state=0)
ads.head()

**Paso 2 ▶️ — La línea base.** Antes de cualquier modelo: ¿cuánto nos equivocamos si siempre predecimos **el promedio** de ventas? (Lección 4.7)

In [ ]:
base = DummyRegressor(strategy="mean").fit(X_ad_tr, y_ad_tr)
print(f"Línea base 'el promedio' → MAE en prueba: {mean_absolute_error(y_ad_te, base.predict(X_ad_te)):.2f} miles de unidades")

**Paso 3 🔮 — La regresión lineal.** Entrenamos con los tres medios. Antes de ejecutar: ¿qué medio crees que tendrá el coeficiente más cercano a cero?

✍️ *Mi apuesta:*

In [ ]:
lineal = LinearRegression().fit(X_ad_tr, y_ad_tr)
pred = lineal.predict(X_ad_te)
print(f"MAE en prueba: {mean_absolute_error(y_ad_te, pred):.2f}   ·   R² en prueba: {r2_score(y_ad_te, pred):.2f}")
print(f"Intercepto: {lineal.intercept_:.2f}")
print(pd.Series(lineal.coef_, index=X_ad.columns).round(3))

✍️ *¿Qué medio funciona y cuál no?*

<details><summary><b>👀 Ver respuesta</b></summary>

El coeficiente del **periódico** es prácticamente **0**: en estos datos, invertir en periódico no se asocia con más ventas. TV y radio sí. El modelo baja el error a casi una cuarta parte del de la línea base (MAE ≈ 0.95 vs. ≈ 3.5) y explica ≈91 % de la variación ($R^2$ ≈ 0.91).

⚠️ Ojo con comparar TV (≈0.04) contra radio (≈0.11) por el tamaño del coeficiente: el rango de inversión en TV es mucho mayor (0–300 vs. 0–50).
</details>

**Paso 4 ✋ — Diagnóstico con los residuos.** Un residuo es `real − predicho`. Si el modelo captó bien la forma de los datos, los residuos deberían verse como una nube sin patrón. Ejecuta y describe lo que ves.

In [ ]:
residuos = y_ad_tr - lineal.predict(X_ad_tr)
plt.scatter(X_ad_tr.tv, residuos, s=12)
plt.axhline(0, color="r")
plt.xlabel("Inversión en TV"); plt.ylabel("Residuo (real − predicho)"); plt.title("¿Hay un patrón?"); plt.show()

✍️ *¿Qué patrón ves y qué significa?*

<details><summary><b>👀 Ver respuesta</b></summary>

Los residuos forman un **arco**: negativos con TV muy baja y muy alta, positivos en medio. El modelo **sobreestima** en los extremos y **subestima** en el centro. Significa que el efecto de la TV **no es una recta**: los primeros miles rinden mucho y luego el efecto se va **saturando** (rendimientos decrecientes). Es un subajuste parcial.
</details>

**Paso 5 ✏️ — Crea una característica.** Un efecto que se satura se parece a una **raíz cuadrada**. Agrega la columna `raiz_tv` y vuelve a entrenar. (✏️ Si quieres, prueba también quitando `periodico`.)

In [ ]:
def preparar(X):
    X = X.copy()
    X["raiz_tv"] = np.sqrt(X.tv)       # ✏️ prueba también np.log1p(X.tv)
    return X                            # ✏️ o:  return X.drop(columns="periodico")

lineal2 = LinearRegression().fit(preparar(X_ad_tr), y_ad_tr)
pred2 = lineal2.predict(preparar(X_ad_te))
print(f"MAE en prueba: {mean_absolute_error(y_ad_te, pred2):.2f}   ·   R² en prueba: {r2_score(y_ad_te, pred2):.2f}")
print(pd.Series(lineal2.coef_, index=preparar(X_ad_tr).columns).round(3))

<details><summary><b>👀 Ver respuesta</b></summary>

El $R^2$ en prueba sube de ≈0.91 a ≈0.93 y el MAE baja de ≈0.95 a ≈0.83. La nueva característica `raiz_tv` se lleva casi todo el efecto de la TV (coeficiente ≈ 1.0, cerca del 0.9 de la regla real) y la columna `tv` original queda casi en 0. Sigue siendo una **regresión lineal**: solo le dimos una característica mejor.
</details>

**Paso 6 ✋ — Responde al gerente.** Usando el modelo del Paso 5:
1. Si la próxima semana invertimos **10 mil $ más en radio** (y lo demás igual), ¿cuántas ventas más esperamos?
2. ¿Le recomendarías seguir invirtiendo en periódico?
3. El gerente pregunta: *"¿Y si invertimos 1,000 mil $ en TV?"* (el máximo histórico es 300). ¿Qué le dirías?

✍️ *Mis respuestas:*

<details><summary><b>👀 Ver respuesta</b></summary>

1. Coeficiente de radio ≈ 0.115 → 10 × 0.115 ≈ **1.2 mil unidades** más, si la relación es causal y se mantiene.
2. Los datos no muestran efecto del periódico (coeficiente ≈ 0). Podrías **probar** reducirlo, idealmente con un experimento controlado: un coeficiente es una asociación, no una prueba de causa.
3. Que es **extrapolar** muy lejos del rango visto (Sección 4.3): el modelo seguirá la curva de la raíz, pero nadie sabe si la saturación continúa igual. Recomendarías subir de forma gradual y medir.
</details>

---
## 📚 Para profundizar
- 🎮 **Interactivo:** *Seeing Theory* (Brown University), capítulo *Regression Analysis*: mueve los puntos y mira cómo cambia la recta.
- 📺 **Video:** StatQuest, "Linear Regression, Clearly Explained" (inglés, con subtítulos).
- 📘 **Documentación:** scikit-learn, guía de usuario, sección *Linear Models* (Ordinary Least Squares, Ridge).

---
⬅️ **Anterior:** [4.9 Proyecto del módulo: Auditoría de un detector de fraude 🕵️](../m04-evaluacion/4.9-proyecto-del-modulo.md) · ➡️ **Siguiente:** [5.2 Regresión logística: de la recta a la probabilidad](5.2-regresion-logistica.ipynb)